# Banking Signature Authentication System
## Secure Login via Handwritten Signature Verification

Professional signature-based authentication system similar to banking security.



### **PROJECT OWNER DETAILS**

| Detail | Information |
|--------|-------------|
| **Name** | Sanusi Shafii |
| **Matriculation Number** | CSA\2023\27683 |
| **Email Address** | s.shafii27683@fudutsinma.edu.ng.com |
| **Phone Number** | +2349125006811 |
| **Department** | Computer Science and Information Technology |
| **Faculty** | Computing |
| **University** | Federal University Dutsin-Ma |
| **Location** | Katsina State, Nigeria |

---

**Project Duration:** November 2023 - November 2024

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import random
import pickle
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Lambda, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import backend as K

from google.colab import files, drive
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

print('Banking Signature Authentication System')
print('TensorFlow Version:', tf.__version__)
print('GPU Available:', len(tf.config.list_physical_devices('GPU')) > 0)

## 2. Mount Google Drive and Dataset Setup

### Dataset Structure:
```
signature_dataset/
    John_Doe/
        sig1.png
        sig2.png
        sig3.png
    Jane_Smith/
        sig1.png
        sig2.png
        sig3.png
    Michael_Brown/
        sig1.png
        sig2.png
```

**Folder name = Person's name (e.g., John_Doe, Jane_Smith)**

In [ ]:
drive.mount('/content/drive')

# Set your dataset path
DATASET_PATH = '/content/drive/MyDrive/signature_dataset'

print(f'Dataset Path: {DATASET_PATH}')
print(f'Path exists: {os.path.exists(DATASET_PATH)}')

## 3. System Configuration

In [ ]:
# Image parameters
IMG_HEIGHT = 128
IMG_WIDTH = 128
IMG_CHANNELS = 1

# Training parameters
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 0.0001

# Verification threshold (similarity score)
SIMILARITY_THRESHOLD = 0.65

# Random seeds
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print('System Configuration Loaded')
print(f'Image Size: {IMG_HEIGHT}x{IMG_WIDTH}')
print(f'Similarity Threshold: {SIMILARITY_THRESHOLD}')

## 4. Load User Signatures

In [ ]:
def load_user_signatures(dataset_path, img_height=128, img_width=128):
    """
    Load all user signatures from dataset.
    Folder name = User name
    """
    user_database = {}
    all_images = []
    all_labels = []
    user_names = []
    
    if not os.path.exists(dataset_path):
        print(f'Error: Dataset path does not exist: {dataset_path}')
        return None, None, None, None
    
    # Get all user folders
    users = sorted([d for d in os.listdir(dataset_path) 
                   if os.path.isdir(os.path.join(dataset_path, d))])
    
    if len(users) == 0:
        print('Error: No user folders found in dataset path!')
        return None, None, None, None
    
    print(f'Found {len(users)} registered users\n')
    
    for user_idx, user_name in enumerate(users):
        user_path = os.path.join(dataset_path, user_name)
        user_signatures = []
        
        # Load all signature images for this user
        image_files = [f for f in os.listdir(user_path) 
                      if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
        
        for img_file in image_files:
            img_path = os.path.join(user_path, img_file)
            
            # Read and preprocess image
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                continue
            
            # Resize and normalize
            img = cv2.resize(img, (img_width, img_height))
            img = img.astype('float32') / 255.0
            
            user_signatures.append(img)
            all_images.append(img)
            all_labels.append(user_idx)
        
        if len(user_signatures) > 0:
            user_database[user_name] = user_signatures
            user_names.append(user_name)
            print(f'{user_idx + 1}. {user_name}: {len(user_signatures)} signatures loaded')
        else:
            print(f'{user_idx + 1}. {user_name}: WARNING - No valid signatures found!')
    
    # Convert to numpy arrays
    all_images = np.array(all_images)
    all_labels = np.array(all_labels)
    
    # Reshape for CNN
    all_images = all_images.reshape(-1, img_height, img_width, 1)
    
    print(f'\nTotal signatures loaded: {len(all_images)}')
    print(f'Total users registered: {len(user_database)}')
    print(f'Image shape: {all_images.shape}')
    
    return user_database, all_images, all_labels, user_names

# Load the dataset
user_database, X, y, user_names = load_user_signatures(DATASET_PATH, IMG_HEIGHT, IMG_WIDTH)

if user_database is not None:
    NUM_USERS = len(user_names)
    print(f'\nRegistered Users: {user_names}')

## 5. Visualize Sample Signatures

In [ ]:
if user_database is not None:
    # Display sample signatures from each user
    num_users_to_show = min(5, len(user_names))
    
    plt.figure(figsize=(15, num_users_to_show * 2))
    
    for i, user_name in enumerate(user_names[:num_users_to_show]):
        signatures = user_database[user_name]
        num_sigs_to_show = min(5, len(signatures))
        
        for j in range(num_sigs_to_show):
            plt.subplot(num_users_to_show, 5, i * 5 + j + 1)
            plt.imshow(signatures[j], cmap='gray')
            plt.title(f'{user_name}\nSig {j+1}')
            plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # User distribution
    signature_counts = [len(sigs) for sigs in user_database.values()]
    
    plt.figure(figsize=(12, 5))
    plt.bar(user_names, signature_counts, color='steelblue', edgecolor='black')
    plt.xlabel('User Name', fontsize=12)
    plt.ylabel('Number of Signatures', fontsize=12)
    plt.title('Signature Count per User', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Create Training Pairs (Siamese Network)

In [ ]:
def create_pairs(user_database, user_names):
    """
    Create pairs of signatures for training:
    - Positive pairs: same user (label = 1)
    - Negative pairs: different users (label = 0)
    """
    pairs = []
    labels = []
    
    # Create positive pairs (same user)
    for user_name in user_names:
        signatures = user_database[user_name]
        
        # Create all combinations of same user signatures
        for i in range(len(signatures)):
            for j in range(i + 1, len(signatures)):
                pairs.append([signatures[i], signatures[j]])
                labels.append(1)
    
    # Create negative pairs (different users)
    for i, user1 in enumerate(user_names):
        for user2 in user_names[i + 1:]:
            sigs1 = user_database[user1]
            sigs2 = user_database[user2]
            
            # Sample some pairs to balance dataset
            num_negative_pairs = min(len(sigs1) * len(sigs2), 5)
            
            for _ in range(num_negative_pairs):
                sig1 = random.choice(sigs1)
                sig2 = random.choice(sigs2)
                pairs.append([sig1, sig2])
                labels.append(0)
    
    return np.array(pairs), np.array(labels)

if user_database is not None:
    pairs, labels = create_pairs(user_database, user_names)
    
    print(f'Total pairs created: {len(pairs)}')
    print(f'Positive pairs (same user): {sum(labels)}')
    print(f'Negative pairs (different users): {len(labels) - sum(labels)}')
    print(f'Pair shape: {pairs.shape}')

## 7. Prepare Training Data

In [ ]:
if user_database is not None:
    # Reshape pairs
    pairs = pairs.reshape(-1, 2, IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)
    
    # Split into train and validation
    indices = np.arange(len(pairs))
    train_indices, val_indices = train_test_split(indices, test_size=0.2, random_state=42)
    
    train_pairs = pairs[train_indices]
    train_labels = labels[train_indices]
    val_pairs = pairs[val_indices]
    val_labels = labels[val_indices]
    
    print(f'Training pairs: {len(train_pairs)}')
    print(f'Validation pairs: {len(val_pairs)}')

## 8. Build Siamese Neural Network

In [ ]:
def build_feature_extractor(input_shape):
    """
    CNN for extracting signature features
    """
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        Conv2D(256, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        GlobalAveragePooling2D(),
        
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(128, activation='relu')
    ])
    
    return model

def cosine_similarity(vectors):
    """
    Compute cosine similarity between feature vectors
    """
    x, y = vectors
    x = K.l2_normalize(x, axis=-1)
    y = K.l2_normalize(y, axis=-1)
    return K.sum(x * y, axis=-1, keepdims=True)

def contrastive_loss(y_true, y_pred, margin=1.0):
    """
    Contrastive loss for Siamese network
    """
    y_pred = K.clip(y_pred, -1.0, 1.0)
    square_pred = K.square(1 - y_pred)
    margin_square = K.square(K.maximum(y_pred - margin, 0))
    return K.mean(y_true * square_pred + (1 - y_true) * margin_square)

if user_database is not None:
    # Build Siamese Network
    input_shape = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)
    
    # Create base network
    base_network = build_feature_extractor(input_shape)
    
    # Create Siamese architecture
    input_a = Input(shape=input_shape, name='input_a')
    input_b = Input(shape=input_shape, name='input_b')
    
    # Process both inputs through same network
    processed_a = base_network(input_a)
    processed_b = base_network(input_b)
    
    # Calculate cosine similarity
    similarity = Lambda(cosine_similarity, name='similarity')([processed_a, processed_b])
    
    # Create model
    siamese_model = Model(inputs=[input_a, input_b], outputs=similarity)
    
    # Compile
    siamese_model.compile(
        optimizer=Adam(LEARNING_RATE),
        loss=contrastive_loss,
        metrics=['accuracy']
    )
    
    print('Siamese Network Built Successfully\n')
    siamese_model.summary()

## 9. Setup Training Callbacks

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'best_banking_signature_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

print('Training callbacks configured')

## 10. Train the Model

In [ ]:
if user_database is not None:
    # Prepare data
    train_img1 = train_pairs[:, 0]
    train_img2 = train_pairs[:, 1]
    val_img1 = val_pairs[:, 0]
    val_img2 = val_pairs[:, 1]
    
    print('Starting training...\n')
    
    # Train
    history = siamese_model.fit(
        [train_img1, train_img2],
        train_labels,
        validation_data=([val_img1, val_img2], val_labels),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )
    
    print('\nTraining completed successfully!')

## 11. Training Visualization

In [ ]:
if user_database is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Training', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Accuracy', fontsize=12)
    axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Loss
    axes[1].plot(history.history['loss'], label='Training', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Loss', fontsize=12)
    axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 12. Save Model and Database

In [ ]:
if user_database is not None:
    # Save models
    siamese_model.save('banking_signature_siamese.h5')
    base_network.save('banking_feature_extractor.h5')
    
    # Save to Google Drive
    siamese_model.save('/content/drive/MyDrive/banking_signature_siamese.h5')
    base_network.save('/content/drive/MyDrive/banking_feature_extractor.h5')
    
    # Save user database
    with open('user_database.pkl', 'wb') as f:
        pickle.dump(user_database, f)
    
    with open('/content/drive/MyDrive/user_database.pkl', 'wb') as f:
        pickle.dump(user_database, f)
    
    print('Models saved successfully!')
    print('User database saved!')

## 13. Authentication Functions

In [ ]:
def authenticate_user(uploaded_img, user_database, model, threshold=0.65):
    """
    Authenticate user by comparing uploaded signature with all registered users.
    Returns: (authenticated, user_name, confidence, all_scores)
    """
    # Preprocess uploaded image
    if len(uploaded_img.shape) == 3:
        uploaded_img = cv2.cvtColor(uploaded_img, cv2.COLOR_BGR2GRAY)
    
    uploaded_processed = cv2.resize(uploaded_img, (IMG_WIDTH, IMG_HEIGHT))
    uploaded_processed = uploaded_processed.astype('float32') / 255.0
    uploaded_processed = uploaded_processed.reshape(1, IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)
    
    user_scores = {}
    
    # Compare with each user's signatures
    for user_name, signatures in user_database.items():
        similarities = []
        
        for signature in signatures:
            sig_input = signature.reshape(1, IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)
            similarity = model.predict([uploaded_processed, sig_input], verbose=0)[0][0]
            similarities.append(similarity)
        
        # Use maximum similarity for this user
        max_similarity = max(similarities)
        avg_similarity = np.mean(similarities)
        
        user_scores[user_name] = {
            'max_similarity': max_similarity,
            'avg_similarity': avg_similarity,
            'confidence': max_similarity * 100
        }
    
    # Find best match
    best_user = max(user_scores.items(), key=lambda x: x[1]['max_similarity'])
    best_user_name = best_user[0]
    best_score = best_user[1]['max_similarity']
    confidence = best_user[1]['confidence']
    
    # Check if above threshold
    authenticated = best_score >= threshold
    
    return authenticated, best_user_name, confidence, user_scores

print('Authentication function loaded')

## 14. Banking-Style Login Interface

In [ ]:
if user_database is not None:
    # Global variables
    uploaded_signature = None
    uploaded_filename = None
    
    # Output widget
    output = widgets.Output()
    
    # Upload button
    upload_button = widgets.FileUpload(
        accept='image/*',
        multiple=False,
        description='Upload Signature'
    )
    
    # Login button
    login_button = widgets.Button(
        description='Authenticate',
        button_style='primary',
        tooltip='Click to authenticate',
        icon='sign-in'
    )
    
    # Banking interface HTML/CSS
    banking_html = """
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700;800&display=swap');
        
        * {
            font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
        }
        
        .banking-header {
            background: linear-gradient(135deg, #0f172a 0%, #1e293b 50%, #334155 100%);
            padding: 50px;
            border-radius: 20px;
            box-shadow: 0 20px 60px rgba(0,0,0,0.5);
            margin: 30px 0;
            border: 1px solid rgba(255,255,255,0.1);
        }
        
        .banking-title {
            color: #f8fafc;
            text-align: center;
            font-size: 42px;
            font-weight: 800;
            margin-bottom: 10px;
            letter-spacing: -1px;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.3);
        }
        
        .banking-subtitle {
            color: #cbd5e1;
            text-align: center;
            font-size: 18px;
            font-weight: 500;
            margin-bottom: 30px;
        }
        
        .security-badge {
            background: rgba(34, 197, 94, 0.2);
            color: #86efac;
            padding: 8px 20px;
            border-radius: 20px;
            display: inline-block;
            font-size: 13px;
            font-weight: 600;
            border: 1px solid #22c55e;
        }
        
        .result-success {
            background: linear-gradient(135deg, #059669 0%, #10b981 100%);
            padding: 40px;
            border-radius: 20px;
            color: white;
            margin: 30px 0;
            box-shadow: 0 20px 50px rgba(16, 185, 129, 0.4);
            animation: slideInUp 0.6s ease-out;
            border: 2px solid #34d399;
        }
        
        .result-denied {
            background: linear-gradient(135deg, #dc2626 0%, #ef4444 100%);
            padding: 40px;
            border-radius: 20px;
            color: white;
            margin: 30px 0;
            box-shadow: 0 20px 50px rgba(239, 68, 68, 0.4);
            animation: slideInUp 0.6s ease-out;
            border: 2px solid #f87171;
        }
        
        .welcome-header {
            font-size: 48px;
            font-weight: 800;
            text-align: center;
            margin: 25px 0;
            text-shadow: 3px 3px 6px rgba(0,0,0,0.3);
            letter-spacing: -1px;
        }
        
        .status-icon {
            font-size: 80px;
            text-align: center;
            margin: 20px 0;
            filter: drop-shadow(0 4px 8px rgba(0,0,0,0.2));
        }
        
        .info-card {
            background: rgba(255,255,255,0.15);
            padding: 25px;
            border-radius: 15px;
            margin: 20px 0;
            backdrop-filter: blur(10px);
            border: 1px solid rgba(255,255,255,0.2);
        }
        
        .info-row {
            display: flex;
            justify-content: space-between;
            padding: 12px 0;
            border-bottom: 1px solid rgba(255,255,255,0.1);
            font-size: 16px;
        }
        
        .info-row:last-child {
            border-bottom: none;
        }
        
        .info-label {
            font-weight: 600;
            opacity: 0.9;
        }
        
        .info-value {
            font-weight: 700;
        }
        
        .confidence-meter {
            background: rgba(255,255,255,0.2);
            border-radius: 10px;
            overflow: hidden;
            height: 40px;
            margin: 20px 0;
            border: 1px solid rgba(255,255,255,0.3);
        }
        
        .confidence-fill {
            height: 100%;
            background: linear-gradient(90deg, rgba(255,255,255,0.9) 0%, rgba(255,255,255,0.7) 100%);
            display: flex;
            align-items: center;
            justify-content: center;
            font-weight: 800;
            font-size: 18px;
            color: #059669;
            transition: width 0.8s cubic-bezier(0.4, 0, 0.2, 1);
            text-shadow: 1px 1px 2px rgba(0,0,0,0.2);
        }
        
        .timestamp {
            text-align: center;
            font-size: 14px;
            opacity: 0.8;
            margin-top: 15px;
            font-weight: 500;
        }
        
        @keyframes slideInUp {
            from {
                opacity: 0;
                transform: translateY(30px);
            }
            to {
                opacity: 1;
                transform: translateY(0);
            }
        }
    </style>
    
    <div class="banking-header">
        <div class="banking-title">SECURE BANKING PORTAL</div>
        <div class="banking-subtitle">Signature-Based Authentication System</div>
        <div style="text-align: center;">
            <span class="security-badge">🔒 256-BIT ENCRYPTED</span>
        </div>
    </div>
    """
    
    display(HTML(banking_html))
    
    def on_upload_change(change):
        global uploaded_signature, uploaded_filename
        with output:
            clear_output(wait=True)
            if len(upload_button.value) > 0:
                uploaded_content = list(upload_button.value.values())[0]
                uploaded_filename = list(upload_button.value.keys())[0]
                
                image_data = np.frombuffer(uploaded_content['content'], np.uint8)
                uploaded_signature = cv2.imdecode(image_data, cv2.IMREAD_GRAYSCALE)
                
                if uploaded_signature is not None:
                    plt.figure(figsize=(10, 6))
                    plt.imshow(uploaded_signature, cmap='gray')
                    plt.title(f'Signature Uploaded: {uploaded_filename}', fontsize=14, fontweight='bold')
                    plt.axis('off')
                    plt.tight_layout()
                    plt.show()
                    print('✓ Signature uploaded successfully')
                    print('Click AUTHENTICATE button to proceed')
                else:
                    print('✗ Error: Could not read the uploaded image')
    
    def on_login_click(b):
        global uploaded_signature, uploaded_filename
        with output:
            if uploaded_signature is None:
                print('⚠ Please upload a signature first!')
                return
            
            clear_output(wait=True)
            print('🔄 Authenticating signature...')
            print('Please wait...')
            
            # Authenticate
            authenticated, user_name, confidence, all_scores = authenticate_user(
                uploaded_signature, user_database, siamese_model, SIMILARITY_THRESHOLD
            )
            
            clear_output(wait=True)
            
            # Get current timestamp
            timestamp = datetime.now().strftime('%B %d, %Y at %I:%M:%S %p')
            
            if authenticated:
                # Format user name (replace underscores with spaces)
                display_name = user_name.replace('_', ' ').title()
                
                result_html = f"""
                <div class="result-success">
                    <div class="status-icon">✓</div>
                    <div class="welcome-header">WELCOME BACK,<br>{display_name.upper()}!</div>
                    
                    <div class="confidence-meter">
                        <div class="confidence-fill" style="width: {confidence}%;">
                            {confidence:.1f}%
                        </div>
                    </div>
                    
                    <div class="info-card">
                        <div class="info-row">
                            <span class="info-label">Status:</span>
                            <span class="info-value">VERIFIED ✓</span>
                        </div>
                        <div class="info-row">
                            <span class="info-label">User ID:</span>
                            <span class="info-value">{user_name}</span>
                        </div>
                        <div class="info-row">
                            <span class="info-label">Authentication:</span>
                            <span class="info-value">SUCCESSFUL</span>
                        </div>
                        <div class="info-row">
                            <span class="info-label">Confidence Score:</span>
                            <span class="info-value">{confidence:.2f}%</span>
                        </div>
                        <div class="info-row">
                            <span class="info-label">Access Level:</span>
                            <span class="info-value">GRANTED</span>
                        </div>
                    </div>
                    
                    <div class="timestamp">Authenticated on {timestamp}</div>
                </div>
                """
            else:
                result_html = f"""
                <div class="result-denied">
                    <div class="status-icon">✗</div>
                    <div class="welcome-header">ACCESS DENIED</div>
                    
                    <div class="confidence-meter">
                        <div class="confidence-fill" style="width: {confidence}%; background: linear-gradient(90deg, rgba(255,255,255,0.9) 0%, rgba(255,255,255,0.7) 100%); color: #dc2626;">
                            {confidence:.1f}%
                        </div>
                    </div>
                    
                    <div class="info-card">
                        <div class="info-row">
                            <span class="info-label">Status:</span>
                            <span class="info-value">NOT VERIFIED ✗</span>
                        </div>
                        <div class="info-row">
                            <span class="info-label">Reason:</span>
                            <span class="info-value">Signature Mismatch</span>
                        </div>
                        <div class="info-row">
                            <span class="info-label">Best Match:</span>
                            <span class="info-value">{user_name} ({confidence:.1f}%)</span>
                        </div>
                        <div class="info-row">
                            <span class="info-label">Required Threshold:</span>
                            <span class="info-value">{SIMILARITY_THRESHOLD * 100:.0f}%</span>
                        </div>
                        <div class="info-row">
                            <span class="info-label">Access Level:</span>
                            <span class="info-value">DENIED</span>
                        </div>
                    </div>
                    
                    <div class="timestamp">Authentication attempt on {timestamp}</div>
                    
                    <div style="text-align: center; margin-top: 20px; padding: 15px; background: rgba(0,0,0,0.2); border-radius: 10px;">
                        <p style="margin: 0; font-size: 14px;">⚠ Please try again or contact your administrator</p>
                    </div>
                </div>
                """
            
            display(HTML(result_html))
            
            # Show signature with result overlay
            plt.figure(figsize=(10, 6))
            plt.imshow(uploaded_signature, cmap='gray')
            if authenticated:
                plt.title(f'✓ VERIFIED: {user_name.replace("_", " ").title()} | Confidence: {confidence:.1f}%',
                         fontsize=16, fontweight='bold', color='green', pad=20)
            else:
                plt.title(f'✗ ACCESS DENIED | Best Match: {user_name} ({confidence:.1f}%) | Required: {SIMILARITY_THRESHOLD*100:.0f}%',
                         fontsize=16, fontweight='bold', color='red', pad=20)
            plt.axis('off')
            plt.tight_layout()
            plt.show()
    
    # Attach handlers
    upload_button.observe(on_upload_change, names='value')
    login_button.on_click(on_login_click)
    
    # Display interface
    display(widgets.HBox([upload_button, login_button]))
    display(output)
    
    print('\n🔐 BANKING AUTHENTICATION SYSTEM READY')
    print('━' * 50)
    print('INSTRUCTIONS:')
    print('1. Click "Upload Signature" to select your signature image')
    print('2. Click "Authenticate" to verify your identity')
    print('━' * 50)
    print(f'Registered Users ({len(user_database)}):')
    for i, user in enumerate(user_names, 1):
        print(f'  {i}. {user.replace("_", " ").title()}')
    print('━' * 50)

## 15. System Performance Summary

In [ ]:
if user_database is not None:
    # Evaluate on validation set
    val_predictions = siamese_model.predict([val_img1, val_img2], verbose=0)
    val_pred_labels = (val_predictions.ravel() >= SIMILARITY_THRESHOLD).astype(int)
    
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    
    accuracy = accuracy_score(val_labels, val_pred_labels)
    precision = precision_score(val_labels, val_pred_labels)
    recall = recall_score(val_labels, val_pred_labels)
    f1 = f1_score(val_labels, val_pred_labels)
    
    summary_html = f"""
    <div style="background: linear-gradient(135deg, #0f172a 0%, #1e293b 50%, #334155 100%);
                padding: 50px; border-radius: 20px; color: white; margin: 40px 0;
                box-shadow: 0 25px 60px rgba(0,0,0,0.6); border: 1px solid rgba(255,255,255,0.1);">
        <h1 style="text-align: center; margin-bottom: 40px; font-size: 40px; font-weight: 800; 
                   letter-spacing: -1px; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">
            BANKING SIGNATURE SYSTEM - DEPLOYED
        </h1>
        
        <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 20px; margin-bottom: 30px;">
            <div style="background: rgba(34, 197, 94, 0.15); padding: 25px; border-radius: 15px; 
                       border: 2px solid #22c55e; backdrop-filter: blur(10px);">
                <h3 style="margin: 0 0 15px 0; color: #86efac; font-size: 14px; font-weight: 600; text-transform: uppercase;">Accuracy</h3>
                <div style="font-size: 48px; font-weight: 800; color: #22c55e;">{accuracy*100:.1f}%</div>
            </div>
            <div style="background: rgba(59, 130, 246, 0.15); padding: 25px; border-radius: 15px; 
                       border: 2px solid #3b82f6; backdrop-filter: blur(10px);">
                <h3 style="margin: 0 0 15px 0; color: #93c5fd; font-size: 14px; font-weight: 600; text-transform: uppercase;">Precision</h3>
                <div style="font-size: 48px; font-weight: 800; color: #3b82f6;">{precision*100:.1f}%</div>
            </div>
            <div style="background: rgba(168, 85, 247, 0.15); padding: 25px; border-radius: 15px; 
                       border: 2px solid #a855f7; backdrop-filter: blur(10px);">
                <h3 style="margin: 0 0 15px 0; color: #d8b4fe; font-size: 14px; font-weight: 600; text-transform: uppercase;">F1 Score</h3>
                <div style="font-size: 48px; font-weight: 800; color: #a855f7;">{f1*100:.1f}%</div>
            </div>
        </div>
        
        <div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 25px;">
            <div style="background: rgba(255,255,255,0.08); padding: 30px; border-radius: 15px; 
                       backdrop-filter: blur(10px); border: 1px solid rgba(255,255,255,0.1);">
                <h2 style="margin-bottom: 20px; color: #fbbf24; font-size: 20px; font-weight: 700;">System Statistics</h2>
                <p style="font-size: 16px; margin: 12px 0; display: flex; justify-content: space-between;">
                    <span style="opacity: 0.9;">Registered Users:</span>
                    <span style="font-weight: 700;">{len(user_database)}</span>
                </p>
                <p style="font-size: 16px; margin: 12px 0; display: flex; justify-content: space-between;">
                    <span style="opacity: 0.9;">Total Signatures:</span>
                    <span style="font-weight: 700;">{len(X)}</span>
                </p>
                <p style="font-size: 16px; margin: 12px 0; display: flex; justify-content: space-between;">
                    <span style="opacity: 0.9;">Training Pairs:</span>
                    <span style="font-weight: 700;">{len(train_pairs):,}</span>
                </p>
                <p style="font-size: 16px; margin: 12px 0; display: flex; justify-content: space-between;">
                    <span style="opacity: 0.9;">Model Type:</span>
                    <span style="font-weight: 700;">Siamese CNN</span>
                </p>
            </div>
            <div style="background: rgba(255,255,255,0.08); padding: 30px; border-radius: 15px; 
                       backdrop-filter: blur(10px); border: 1px solid rgba(255,255,255,0.1);">
                <h2 style="margin-bottom: 20px; color: #34d399; font-size: 20px; font-weight: 700;">Model Performance</h2>
                <p style="font-size: 16px; margin: 12px 0; display: flex; justify-content: space-between;">
                    <span style="opacity: 0.9;">Validation Accuracy:</span>
                    <span style="font-weight: 700;">{accuracy*100:.2f}%</span>
                </p>
                <p style="font-size: 16px; margin: 12px 0; display: flex; justify-content: space-between;">
                    <span style="opacity: 0.9;">Recall:</span>
                    <span style="font-weight: 700;">{recall*100:.2f}%</span>
                </p>
                <p style="font-size: 16px; margin: 12px 0; display: flex; justify-content: space-between;">
                    <span style="opacity: 0.9;">Training Epochs:</span>
                    <span style="font-weight: 700;">{len(history.history['loss'])}</span>
                </p>
                <p style="font-size: 16px; margin: 12px 0; display: flex; justify-content: space-between;">
                    <span style="opacity: 0.9;">Threshold:</span>
                    <span style="font-weight: 700;">{SIMILARITY_THRESHOLD*100:.0f}%</span>
                </p>
            </div>
        </div>
        
        <div style="margin-top: 30px; padding: 25px; background: rgba(255,255,255,0.05); 
                   border-radius: 15px; border: 1px solid rgba(255,255,255,0.1);">
            <h2 style="margin-bottom: 15px; color: #a78bfa; font-size: 20px; font-weight: 700; text-align: center;">System Capabilities</h2>
            <div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 15px; font-size: 15px;">
                <div>✓ Real-time signature authentication</div>
                <div>✓ Multi-user recognition system</div>
                <div>✓ Confidence-based verification</div>
                <div>✓ Banking-grade security interface</div>
                <div>✓ Automated identity verification</div>
                <div>✓ Professional access control</div>
            </div>
        </div>
        
        <div style="text-align: center; margin-top: 35px; padding: 20px; 
                   background: linear-gradient(135deg, rgba(251, 191, 36, 0.15) 0%, rgba(245, 158, 11, 0.15) 100%); 
                   border-radius: 12px; border: 2px solid #fbbf24;">
            <h3 style="color: #fbbf24; font-size: 24px; margin-bottom: 10px; font-weight: 800;">4th Year Project</h3>
            <p style="font-size: 16px; margin: 0; font-weight: 600;">Handwritten Signature Verification System</p>
            <p style="font-size: 14px; margin: 8px 0 0 0; opacity: 0.9;">Deep Learning | CNN | Siamese Networks | Biometric Authentication</p>
        </div>
    </div>
    """
    
    display(HTML(summary_html))